In [ ]:

import numpy as np


def relu(z):
    return np.maximum(0, z)

def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def one_hot(y, classes=10):
    oh = np.zeros((len(y), classes))
    oh[np.arange(len(y)), y] = 1
    return oh

def accuracy(X, y, W, b):
    A = X
    for i in range(len(W) - 1):
        A = relu(A @ W[i] + b[i])
    A = softmax(A @ W[-1] + b[-1])
    return np.mean(np.argmax(A, axis=1) == y) * 100


np.random.seed(0)
X_all, y_all = [], []
for digit in range(10):
    mean = np.zeros(784)
    mean[digit*78 : digit*78+78] = 0.7  
    X_class = np.clip(mean + np.random.randn(600, 784) * 0.15, 0, 1)
    X_all.append(X_class)
    y_all.append(np.full(600, digit))

X = np.vstack(X_all).astype(np.float32)
y = np.concatenate(y_all)
idx = np.random.permutation(len(X))
X, y = X[idx], y[idx]

X_train, y_train = X[:4800], y[:4800]
X_test,  y_test  = X[4800:], y[4800:]

# Weights Initialize 
np.random.seed(42)
sizes = [784, 128, 64, 10]
W = [np.random.randn(sizes[i], sizes[i+1]) * np.sqrt(2/sizes[i])
     for i in range(len(sizes)-1)]
b = [np.zeros((1, sizes[i+1])) for i in range(len(sizes)-1)]

# Training Loop
Y_train = one_hot(y_train)
lr, epochs, batch = 0.05, 15, 64

print(f"{'Epoch':>6} | {'Train Acc':>10} | {'Test Acc':>9} | {'Loss':>8}")
print("-" * 45)

for ep in range(1, epochs + 1):
    idx = np.random.permutation(len(X_train))
    Xs, Ys = X_train[idx], Y_train[idx]

    for s in range(0, len(X_train), batch):
        Xb, Yb = Xs[s:s+batch], Ys[s:s+batch]

        # Forward Pass
        A, Z = [Xb], []
        for i in range(len(W) - 1):
            z = A[-1] @ W[i] + b[i];  Z.append(z);  A.append(relu(z))
        z = A[-1] @ W[-1] + b[-1];    Z.append(z);  A.append(softmax(z))

        # Backward Pass
        delta = A[-1] - Yb
        for i in reversed(range(len(W))):
            dW = (A[i].T @ delta) / len(Xb)
            db = delta.mean(axis=0, keepdims=True)
            W[i] -= lr * dW
            b[i] -= lr * db
            if i > 0:
                delta = (delta @ W[i].T) * (Z[i-1] > 0)

    # Loss Calculate Karo
    A_e = X_train
    for i in range(len(W)-1): A_e = relu(A_e @ W[i] + b[i])
    A_e = softmax(A_e @ W[-1] + b[-1])
    loss = -np.mean(np.sum(Y_train * np.log(A_e + 1e-8), axis=1))

    tr = accuracy(X_train, y_train, W, b)
    te = accuracy(X_test,  y_test,  W, b)
    print(f"  {ep:4d}  |  {tr:7.2f}%  |  {te:7.2f}%  | {loss:8.4f}")

print(f"\nFinal Test Accuracy: {accuracy(X_test, y_test, W, b):.2f}%")

 Epoch |  Train Acc |  Test Acc |     Loss
---------------------------------------------
     1  |   100.00%  |   100.00%  |   0.0356
     2  |   100.00%  |   100.00%  |   0.0126
     3  |   100.00%  |   100.00%  |   0.0072
     4  |   100.00%  |   100.00%  |   0.0050
     5  |   100.00%  |   100.00%  |   0.0037
     6  |   100.00%  |   100.00%  |   0.0030
     7  |   100.00%  |   100.00%  |   0.0025
     8  |   100.00%  |   100.00%  |   0.0021
     9  |   100.00%  |   100.00%  |   0.0018
    10  |   100.00%  |   100.00%  |   0.0016
    11  |   100.00%  |   100.00%  |   0.0014
    12  |   100.00%  |   100.00%  |   0.0013
    13  |   100.00%  |   100.00%  |   0.0012
    14  |   100.00%  |   100.00%  |   0.0011
    15  |   100.00%  |   100.00%  |   0.0010

Final Test Accuracy: 100.00%
